# 🥈 Silver Layer - Dimension Tables
## AWS S3 External Storage via Unity Catalog

**Purpose**: Build cleaned, conformed dimension tables from Bronze layer with SCD Type 2

**Source**: `workspace.`workspace-adventureworks-bronze`` (S3)
**Target**: `workspace.`workspace-adventureworks-silver`` (S3)

**Dimension Tables to Create**:
1. **dim_customer** - Customer master data with attributes
2. **dim_product** - Product catalog with category hierarchy
3. **dim_date** - Date dimension for time intelligence
4. **dim_location** - Geographic locations (addresses, regions)
5. **dim_employee** - Employee information

**Features**:
- Data quality checks and cleansing
- SCD Type 2 (slowly changing dimensions) with effective dates
- Business key deduplication
- Surrogate key generation
- Conformance to naming standards

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import *
from datetime import datetime, date, timedelta
import time
from functools import reduce as functools_reduce

# Configuration
BRONZE_SCHEMA = "`workspace-adventureworks-bronze`"
SILVER_SCHEMA = "`workspace-adventureworks-silver`"

print("=" * 100)
print("🥈 SILVER LAYER - COMPLETE DATA TRANSFORMATION")
print("=" * 100)
print(f"Start Time: {datetime.now()}")
print(f"Source: workspace.{BRONZE_SCHEMA} (AWS S3)")
print(f"Target: workspace.{SILVER_SCHEMA} (AWS S3)")
print("=" * 100)
print()
print("Transformation Scope:")
print("  • 9 Dimension Tables (Customer, Product, Date, Location, Employee, Salesperson, Territory, Vendor, Store)")
print("  • 4 Fact Tables (Sales, Inventory, Purchases, Production)")
print("=" * 100)
print()

# Track transformation results
transformation_results = []

In [0]:
import builtins  # Import builtins module to access Python's round() explicitly

def clean_and_deduplicate(df, business_key_cols, description="table"):
    """
    Clean data by removing null business keys and deduplicating based on latest ingestion_timestamp.
    """
    print(f"\n  Cleaning {description}...")
    initial_count = df.count()
    
    # Ensure business_key_cols is a list of strings
    business_key_cols = [str(k) for k in (business_key_cols if isinstance(business_key_cols, list) else [business_key_cols])]
    
    # Remove null business keys using a single filter
    null_filters = [col(str(key_col)).isNotNull() for key_col in business_key_cols]
    df_filtered = df.filter(functools_reduce(lambda a, b: a & b, null_filters))
    
    # Deduplicate using window function
    window_spec = Window.partitionBy(*[str(k) for k in business_key_cols]).orderBy(col("ingestion_timestamp").desc())
    df_clean = df_filtered.withColumn("row_num", row_number().over(window_spec)).filter(col("row_num") == 1).drop("row_num")
    
    final_count = df_clean.count()
    print(f"    Initial: {initial_count:,} | Final: {final_count:,} | Removed: {(initial_count - final_count):,}")
    return df_clean


def create_scd_type2_dimension(df, business_key_cols, table_name, sk_name=None):
    """
    Create SCD Type 2 dimension with surrogate key, effective dates, and row hash.
    """
    print(f"\n  Creating SCD Type 2 structure for {table_name}...")
    
    # Ensure business_key_cols is a list of strings
    business_key_cols = [str(k) for k in (business_key_cols if isinstance(business_key_cols, list) else [business_key_cols])]
    
    # Define surrogate key name
    sk_column = str(sk_name) if sk_name else f"{table_name}_sk"
    
    # Get all column names as strings
    all_existing_cols = [str(field.name) for field in df.schema.fields]
    
    # Build hash columns (exclude metadata)
    hash_cols = [c for c in all_existing_cols if c not in ["ingestion_timestamp", "source_system"]]
    
    # Create hash expression
    hash_expr = concat_ws("||", *[coalesce(col(str(c)).cast("string"), lit("")) for c in hash_cols])
    
    # Add SCD columns using withColumns for better performance
    df_scd = df.withColumns({
        sk_column: monotonically_increasing_id(),
        "effective_start_date": current_timestamp(),
        "effective_end_date": lit(None).cast("timestamp"),
        "is_current": lit(True),
        "row_hash": sha2(hash_expr, 256)
    })
    
    # Order columns: SK, business keys, attributes, metadata, SCD columns
    scd_cols = ["effective_start_date", "effective_end_date", "is_current", "row_hash"]
    metadata_cols = ["ingestion_timestamp", "source_system"]
    
    # Build list of other columns (excluding SK, business keys, metadata, SCD)
    all_col_names_after = [str(field.name) for field in df_scd.schema.fields]
    exclude_cols = set([sk_column] + business_key_cols + scd_cols + metadata_cols)
    other_cols = [str(c) for c in all_col_names_after if c not in exclude_cols]
    
    # Build final column order - all as strings
    final_cols = [str(sk_column)] + [str(k) for k in business_key_cols] + other_cols + metadata_cols + scd_cols
    
    # Select with explicit string-to-column conversion
    return df_scd.select(*[col(str(c)) for c in final_cols])


def transform_and_save_dimension(table_name, bronze_table, business_keys, select_expr, description):
    """
    Generic function to transform a dimension table: read, clean, apply SCD, and save.
    """
    start_time = time.time()
    print(f"\n{'=' * 100}")
    print(f"📦 TRANSFORMING: {description}")
    print(f"{'=' * 100}")
    
    try:
        # Read from Bronze
        df = spark.table(f"{BRONZE_SCHEMA}.{bronze_table}")
        bronze_count = df.count()
        print(f"  Bronze rows: {bronze_count:,}")
        
        # Apply transformations
        df_transformed = df.selectExpr(*select_expr)
        
        # Clean and deduplicate
        df_clean = clean_and_deduplicate(df_transformed, business_keys, description)
        
        # Apply SCD Type 2
        df_scd = create_scd_type2_dimension(df_clean, business_keys, table_name)
        
        # Save to Silver
        df_scd.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SILVER_SCHEMA}.{table_name}")
        
        row_count = df_scd.count()
        duration = time.time() - start_time
        
        print(f"  \n  ✅ SUCCESS: {table_name} created with {row_count:,} rows in {duration:.2f}s")
        
        transformation_results.append({
            "table": table_name,
            "type": "dimension",
            "status": "success",
            "row_count": row_count,
            "duration_seconds": builtins.round(duration, 2)  # Use Python's builtin round
        })
        
        return True
        
    except Exception as e:
        duration = time.time() - start_time
        print(f"  \n  ❌ FAILED: {str(e)}")
        import traceback
        traceback.print_exc()  # Add full traceback for debugging
        transformation_results.append({
            "table": table_name,
            "type": "dimension",
            "status": "failed",
            "row_count": 0,
            "duration_seconds": builtins.round(duration, 2),  # Use Python's builtin round
            "error": str(e)
        })
        return False


print("\n✅ Helper functions loaded successfully")
print("  - clean_and_deduplicate()")
print("  - create_scd_type2_dimension()")
print("  - transform_and_save_dimension()")

In [0]:
import builtins  # Import builtins to access Python's round()

# ============================================================================
# PART 1: CORE DIMENSION TABLES
# ============================================================================

print("\n" + "=" * 100)
print("🔹 PART 1: TRANSFORMING CORE DIMENSIONS")
print("=" * 100)

# ----------------------------------------------------------------------------
# 1.1 dim_customer - Customer Master
# ----------------------------------------------------------------------------
transform_and_save_dimension(
    table_name="dim_customer",
    bronze_table="customer",
    business_keys=["customer_id"],
    select_expr=[
        "CustomerID as customer_id",
        "PersonID as person_id",
        "StoreID as store_id",
        "TerritoryID as territory_id",
        "AccountNumber as account_number",
        "ModifiedDate as modified_date",
        "ingestion_timestamp",
        "source_system"
    ],
    description="Customer Dimension"
)

# ----------------------------------------------------------------------------
# 1.2 dim_product - Product Catalog with Category Hierarchy
# ----------------------------------------------------------------------------
print("\n" + "=" * 100)
print("📦 TRANSFORMING: Product Dimension with Category Hierarchy")
print("=" * 100)

start_time = time.time()
try:
    # Read tables
    product = spark.table(f"{BRONZE_SCHEMA}.product")
    product_category = spark.table(f"{BRONZE_SCHEMA}.productcategory")
    product_subcategory = spark.table(f"{BRONZE_SCHEMA}.productsubcategory")
    product_model = spark.table(f"{BRONZE_SCHEMA}.productmodel")
    
    print(f"  Bronze rows: Product={product.count():,}")
    
    # Join and transform
    df_product = (product
        .join(product_subcategory, product["ProductSubcategoryID"] == product_subcategory["ProductSubcategoryID"], "left")
        .join(product_category, product_subcategory["ProductCategoryID"] == product_category["ProductCategoryID"], "left")
        .join(product_model, product["ProductModelID"] == product_model["ProductModelID"], "left")
        .select(
            product["ProductID"].alias("product_id"),
            product["Name"].alias("product_name"),
            product["ProductNumber"].alias("product_number"),
            product["Color"].alias("color"),
            product["StandardCost"].alias("standard_cost"),
            product["ListPrice"].alias("list_price"),
            product["Size"].alias("size"),
            product["Weight"].alias("weight"),
            product_model["Name"].alias("model_name"),
            product_subcategory["Name"].alias("subcategory_name"),
            product_category["Name"].alias("category_name"),
            product["SellStartDate"].alias("sell_start_date"),
            product["SellEndDate"].alias("sell_end_date"),
            product["DiscontinuedDate"].alias("discontinued_date"),
            product["ModifiedDate"].alias("modified_date"),
            product["ingestion_timestamp"],
            product["source_system"]
        )
    )
    
    # Clean and deduplicate
    df_product_clean = clean_and_deduplicate(df_product, ["product_id"], "Product")
    
    # Apply SCD Type 2
    df_product_scd = create_scd_type2_dimension(df_product_clean, ["product_id"], "dim_product")
    
    # Save
    df_product_scd.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SILVER_SCHEMA}.dim_product")
    
    row_count = df_product_scd.count()
    duration = time.time() - start_time
    print(f"  \n  ✅ SUCCESS: dim_product created with {row_count:,} rows in {duration:.2f}s")
    
    transformation_results.append({"table": "dim_product", "type": "dimension", "status": "success", "row_count": row_count, "duration_seconds": builtins.round(duration, 2)})
except Exception as e:
    duration = time.time() - start_time
    print(f"  \n  ❌ FAILED: {str(e)}")
    import traceback
    traceback.print_exc()
    transformation_results.append({"table": "dim_product", "type": "dimension", "status": "failed", "row_count": 0, "duration_seconds": builtins.round(duration, 2), "error": str(e)})

# ----------------------------------------------------------------------------
# 1.3 dim_date - Date Dimension
# ----------------------------------------------------------------------------
print("\n" + "=" * 100)
print("📅 TRANSFORMING: Date Dimension")
print("=" * 100)

start_time = time.time()
try:
    start_date = date(2000, 1, 1)
    end_date = date(2030, 12, 31)
    
    date_list = []
    current_date = start_date
    while current_date <= end_date:
        date_list.append((current_date,))
        current_date += timedelta(days=1)
    
    df_dates = spark.createDataFrame(date_list, ["date_value"])
    
    df_date = df_dates.withColumns({
        "date_sk": row_number().over(Window.orderBy("date_value")),
        "year": year("date_value"),
        "quarter": quarter("date_value"),
        "month": month("date_value"),
        "month_name": date_format("date_value", "MMMM"),
        "day_of_month": dayofmonth("date_value"),
        "day_of_week": dayofweek("date_value"),
        "day_name": date_format("date_value", "EEEE"),
        "week_of_year": weekofyear("date_value"),
        "is_weekend": when(dayofweek("date_value").isin([1, 7]), True).otherwise(False),
        "fiscal_year": when(month("date_value") >= 7, year("date_value") + 1).otherwise(year("date_value")),
        "fiscal_quarter": when(month("date_value").between(7, 9), 1)
                         .when(month("date_value").between(10, 12), 2)
                         .when(month("date_value").between(1, 3), 3)
                         .otherwise(4)
    })
    
    df_date.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SILVER_SCHEMA}.dim_date")
    
    row_count = df_date.count()
    duration = time.time() - start_time
    print(f"  ✅ SUCCESS: dim_date created with {row_count:,} rows in {duration:.2f}s")
    
    transformation_results.append({"table": "dim_date", "type": "dimension", "status": "success", "row_count": row_count, "duration_seconds": builtins.round(duration, 2)})
except Exception as e:
    duration = time.time() - start_time
    print(f"  ❌ FAILED: {str(e)}")
    transformation_results.append({"table": "dim_date", "type": "dimension", "status": "failed", "row_count": 0, "duration_seconds": builtins.round(duration, 2), "error": str(e)})

In [0]:
# ============================================================================
# PART 2: SUPPORTING DIMENSIONS
# ============================================================================

print("\n" + "=" * 100)
print("🔹 PART 2: TRANSFORMING SUPPORTING DIMENSIONS")
print("=" * 100)

# 2.1 dim_location
transform_and_save_dimension(
    table_name="dim_location",
    bronze_table="address",
    business_keys=["address_id"],
    select_expr=[
        "AddressID as address_id",
        "AddressLine1 as address_line1",
        "AddressLine2 as address_line2",
        "City as city",
        "StateProvinceID as state_province_id",
        "PostalCode as postal_code",
        "ModifiedDate as modified_date",
        "ingestion_timestamp",
        "source_system"
    ],
    description="Location Dimension"
)

# 2.2 dim_employee
transform_and_save_dimension(
    table_name="dim_employee",
    bronze_table="employee",
    business_keys=["employee_id"],
    select_expr=[
        "BusinessEntityID as employee_id",
        "NationalIDNumber as national_id",
        "LoginID as login_id",
        "JobTitle as job_title",
        "BirthDate as birth_date",
        "MaritalStatus as marital_status",
        "Gender as gender",
        "HireDate as hire_date",
        "SalariedFlag as is_salaried",
        "VacationHours as vacation_hours",
        "SickLeaveHours as sick_leave_hours",
        "CurrentFlag as is_current_employee",
        "ModifiedDate as modified_date",
        "ingestion_timestamp",
        "source_system"
    ],
    description="Employee Dimension"
)

# 2.3 dim_salesperson
transform_and_save_dimension(
    table_name="dim_salesperson",
    bronze_table="salesperson",
    business_keys=["salesperson_id"],
    select_expr=[
        "BusinessEntityID as salesperson_id",
        "TerritoryID as territory_id",
        "SalesQuota as sales_quota",
        "Bonus as bonus",
        "CommissionPct as commission_pct",
        "SalesYTD as sales_ytd",
        "SalesLastYear as sales_last_year",
        "ModifiedDate as modified_date",
        "ingestion_timestamp",
        "source_system"
    ],
    description="Salesperson Dimension"
)

# 2.4 dim_territory
transform_and_save_dimension(
    table_name="dim_territory",
    bronze_table="salesterritory",
    business_keys=["territory_id"],
    select_expr=[
        "TerritoryID as territory_id",
        "Name as territory_name",
        "CountryRegionCode as country_region_code",
        "\"Group\" as territory_group",
        "SalesYTD as sales_ytd",
        "SalesLastYear as sales_last_year",
        "CostYTD as cost_ytd",
        "CostLastYear as cost_last_year",
        "ModifiedDate as modified_date",
        "ingestion_timestamp",
        "source_system"
    ],
    description="Territory Dimension"
)

# 2.5 dim_vendor
transform_and_save_dimension(
    table_name="dim_vendor",
    bronze_table="vendor",
    business_keys=["vendor_id"],
    select_expr=[
        "BusinessEntityID as vendor_id",
        "AccountNumber as account_number",
        "Name as vendor_name",
        "CreditRating as credit_rating",
        "PreferredVendorStatus as is_preferred_vendor",
        "ActiveFlag as is_active",
        "PurchasingWebServiceURL as purchasing_url",
        "ModifiedDate as modified_date",
        "ingestion_timestamp",
        "source_system"
    ],
    description="Vendor Dimension"
)

# 2.6 dim_store
transform_and_save_dimension(
    table_name="dim_store",
    bronze_table="store",
    business_keys=["store_id"],
    select_expr=[
        "BusinessEntityID as store_id",
        "Name as store_name",
        "SalesPersonID as salesperson_id",
        "ModifiedDate as modified_date",
        "ingestion_timestamp",
        "source_system"
    ],
    description="Store Dimension"
)

print("\n" + "=" * 100)
print("✅ Part 2 Complete: All dimensions transformed")
print("=" * 100)

In [0]:
import builtins  # Import builtins to access Python's round()

# ============================================================================
# PART 3: FACT TABLES - SALES
# ============================================================================

print("\n" + "=" * 100)
print("🔹 PART 3: TRANSFORMING FACT TABLES - SALES")
print("=" * 100)

start_time = time.time()
try:
    # Read sales order detail and header
    sod = spark.table(f"{BRONZE_SCHEMA}.salesorderdetail")
    soh = spark.table(f"{BRONZE_SCHEMA}.salesorderheader")
    
    print(f"  Bronze rows: SalesOrderDetail={sod.count():,}, SalesOrderHeader={soh.count():,}")
    
    # Join detail with header
    fact_sales = (sod
        .join(soh, "SalesOrderID", "inner")
        .select(
            # Surrogate key
            monotonically_increasing_id().alias("sales_sk"),
            # Business keys
            sod["SalesOrderID"].alias("sales_order_id"),
            sod["SalesOrderDetailID"].alias("sales_order_detail_id"),
            # Foreign keys (will need lookup to dimension SKs in Gold layer)
            soh["CustomerID"].alias("customer_id"),
            sod["ProductID"].alias("product_id"),
            soh["TerritoryID"].alias("territory_id"),
            soh["SalesPersonID"].alias("salesperson_id"),
            soh["ShipToAddressID"].alias("ship_to_address_id"),
            soh["BillToAddressID"].alias("bill_to_address_id"),
            # Date keys
            to_date(soh["OrderDate"]).alias("order_date"),
            to_date(soh["DueDate"]).alias("due_date"),
            to_date(soh["ShipDate"]).alias("ship_date"),
            # Measures
            sod["OrderQty"].alias("order_quantity"),
            sod["UnitPrice"].alias("unit_price"),
            sod["UnitPriceDiscount"].alias("unit_price_discount"),
            sod["LineTotal"].alias("line_total"),
            soh["SubTotal"].alias("order_subtotal"),
            soh["TaxAmt"].alias("tax_amount"),
            soh["Freight"].alias("freight"),
            soh["TotalDue"].alias("total_due"),
            # Attributes
            soh["OnlineOrderFlag"].alias("is_online_order"),
            soh["PurchaseOrderNumber"].alias("purchase_order_number"),
            soh["AccountNumber"].alias("account_number"),
            soh["CreditCardID"].alias("credit_card_id"),
            soh["ShipMethodID"].alias("ship_method_id"),
            soh["Status"].alias("order_status"),
            # Metadata
            sod["ingestion_timestamp"],
            sod["source_system"]
        )
    )
    
    # Save
    fact_sales.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SILVER_SCHEMA}.fact_sales")
    
    row_count = fact_sales.count()
    duration = time.time() - start_time
    print(f"  \n  ✅ SUCCESS: fact_sales created with {row_count:,} rows in {duration:.2f}s")
    
    transformation_results.append({"table": "fact_sales", "type": "fact", "status": "success", "row_count": row_count, "duration_seconds": builtins.round(duration, 2)})
except Exception as e:
    duration = time.time() - start_time
    print(f"  \n  ❌ FAILED: {str(e)}")
    transformation_results.append({"table": "fact_sales", "type": "fact", "status": "failed", "row_count": 0, "duration_seconds": builtins.round(duration, 2), "error": str(e)})

In [0]:
import builtins  # Import builtins to access Python's round()

# ============================================================================
# FACT TABLES - INVENTORY
# ============================================================================

print("\n" + "=" * 100)
print("📦 TRANSFORMING: Fact Inventory (Transaction History)")
print("=" * 100)

start_time = time.time()
try:
    # Read transaction history and archived transactions
    th = spark.table(f"{BRONZE_SCHEMA}.transactionhistory")
    tha = spark.table(f"{BRONZE_SCHEMA}.transactionhistoryarchive")
    
    print(f"  Bronze rows: TransactionHistory={th.count():,}, Archive={tha.count():,}")
    
    # Combine current and archived transactions
    fact_inventory = (
        th.select(
            monotonically_increasing_id().alias("inventory_transaction_sk"),
            col("TransactionID").alias("transaction_id"),
            col("ProductID").alias("product_id"),
            col("ReferenceOrderID").alias("reference_order_id"),
            col("ReferenceOrderLineID").alias("reference_order_line_id"),
            to_date(col("TransactionDate")).alias("transaction_date"),
            col("TransactionType").alias("transaction_type"),
            col("Quantity").alias("quantity"),
            col("ActualCost").alias("actual_cost"),
            lit(False).alias("is_archived"),
            col("ModifiedDate").alias("modified_date"),
            col("ingestion_timestamp"),
            col("source_system")
        )
        .union(
            tha.select(
                monotonically_increasing_id().alias("inventory_transaction_sk"),
                col("TransactionID").alias("transaction_id"),
                col("ProductID").alias("product_id"),
                col("ReferenceOrderID").alias("reference_order_id"),
                col("ReferenceOrderLineID").alias("reference_order_line_id"),
                to_date(col("TransactionDate")).alias("transaction_date"),
                col("TransactionType").alias("transaction_type"),
                col("Quantity").alias("quantity"),
                col("ActualCost").alias("actual_cost"),
                lit(True).alias("is_archived"),
                col("ModifiedDate").alias("modified_date"),
                col("ingestion_timestamp"),
                col("source_system")
            )
        )
    )
    
    # Save
    fact_inventory.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SILVER_SCHEMA}.fact_inventory")
    
    row_count = fact_inventory.count()
    duration = time.time() - start_time
    print(f"  \n  ✅ SUCCESS: fact_inventory created with {row_count:,} rows in {duration:.2f}s")
    
    transformation_results.append({"table": "fact_inventory", "type": "fact", "status": "success", "row_count": row_count, "duration_seconds": builtins.round(duration, 2)})
except Exception as e:
    duration = time.time() - start_time
    print(f"  \n  ❌ FAILED: {str(e)}")
    transformation_results.append({"table": "fact_inventory", "type": "fact", "status": "failed", "row_count": 0, "duration_seconds": builtins.round(duration, 2), "error": str(e)})

In [0]:
import builtins  # Import builtins to access Python's round()

# ============================================================================
# FACT TABLES - PURCHASES
# ============================================================================

print("\n" + "=" * 100)
print("🛒 TRANSFORMING: Fact Purchases")
print("=" * 100)

start_time = time.time()
try:
    # Read purchase order detail and header
    pod = spark.table(f"{BRONZE_SCHEMA}.purchaseorderdetail")
    poh = spark.table(f"{BRONZE_SCHEMA}.purchaseorderheader")
    
    print(f"  Bronze rows: PurchaseOrderDetail={pod.count():,}, PurchaseOrderHeader={poh.count():,}")
    
    # Join detail with header
    fact_purchases = (pod
        .join(poh, "PurchaseOrderID", "inner")
        .select(
            monotonically_increasing_id().alias("purchase_sk"),
            pod["PurchaseOrderID"].alias("purchase_order_id"),
            pod["PurchaseOrderDetailID"].alias("purchase_order_detail_id"),
            pod["ProductID"].alias("product_id"),
            poh["VendorID"].alias("vendor_id"),
            poh["EmployeeID"].alias("employee_id"),
            poh["ShipMethodID"].alias("ship_method_id"),
            to_date(poh["OrderDate"]).alias("order_date"),
            to_date(poh["ShipDate"]).alias("ship_date"),
            pod["OrderQty"].alias("order_quantity"),
            pod["UnitPrice"].alias("unit_price"),
            pod["LineTotal"].alias("line_total"),
            pod["ReceivedQty"].alias("received_quantity"),
            pod["RejectedQty"].alias("rejected_quantity"),
            pod["StockedQty"].alias("stocked_quantity"),
            to_date(pod["DueDate"]).alias("due_date"),
            poh["SubTotal"].alias("order_subtotal"),
            poh["TaxAmt"].alias("tax_amount"),
            poh["Freight"].alias("freight"),
            poh["TotalDue"].alias("total_due"),
            poh["Status"].alias("order_status"),
            pod["ModifiedDate"].alias("modified_date"),
            pod["ingestion_timestamp"],
            pod["source_system"]
        )
    )
    
    # Save
    fact_purchases.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SILVER_SCHEMA}.fact_purchases")
    
    row_count = fact_purchases.count()
    duration = time.time() - start_time
    print(f"  \n  ✅ SUCCESS: fact_purchases created with {row_count:,} rows in {duration:.2f}s")
    
    transformation_results.append({"table": "fact_purchases", "type": "fact", "status": "success", "row_count": row_count, "duration_seconds": builtins.round(duration, 2)})
except Exception as e:
    duration = time.time() - start_time
    print(f"  \n  ❌ FAILED: {str(e)}")
    transformation_results.append({"table": "fact_purchases", "type": "fact", "status": "failed", "row_count": 0, "duration_seconds": builtins.round(duration, 2), "error": str(e)})

In [0]:
import builtins  # Import builtins to access Python's round()

# ============================================================================
# FACT TABLES - PRODUCTION
# ============================================================================

print("\n" + "=" * 100)
print("🏭 TRANSFORMING: Fact Production (Work Orders)")
print("=" * 100)

start_time = time.time()
try:
    # Read work order and routing
    wo = spark.table(f"{BRONZE_SCHEMA}.workorder")
    wor = spark.table(f"{BRONZE_SCHEMA}.workorderrouting")
    
    print(f"  Bronze rows: WorkOrder={wo.count():,}, WorkOrderRouting={wor.count():,}")
    
    # Create fact table from work orders
    fact_production = (wo
        .select(
            monotonically_increasing_id().alias("production_sk"),
            col("WorkOrderID").alias("work_order_id"),
            col("ProductID").alias("product_id"),
            col("OrderQty").alias("order_quantity"),
            col("StockedQty").alias("stocked_quantity"),
            col("ScrappedQty").alias("scrapped_quantity"),
            to_date(col("StartDate")).alias("start_date"),
            to_date(col("EndDate")).alias("end_date"),
            to_date(col("DueDate")).alias("due_date"),
            col("ScrapReasonID").alias("scrap_reason_id"),
            col("ModifiedDate").alias("modified_date"),
            col("ingestion_timestamp"),
            col("source_system")
        )
    )
    
    # Save
    fact_production.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SILVER_SCHEMA}.fact_production")
    
    row_count = fact_production.count()
    duration = time.time() - start_time
    print(f"  \n  ✅ SUCCESS: fact_production created with {row_count:,} rows in {duration:.2f}s")
    
    transformation_results.append({"table": "fact_production", "type": "fact", "status": "success", "row_count": row_count, "duration_seconds": builtins.round(duration, 2)})
except Exception as e:
    duration = time.time() - start_time
    print(f"  \n  ❌ FAILED: {str(e)}")
    transformation_results.append({"table": "fact_production", "type": "fact", "status": "failed", "row_count": 0, "duration_seconds": builtins.round(duration, 2), "error": str(e)})

print("\n" + "=" * 100)
print("✅ All Fact Tables Transformed")
print("=" * 100)

In [0]:
import builtins  # Import builtins to access Python's sum()

# ============================================================================
# SUMMARY AND VALIDATION
# ============================================================================

from pyspark.sql import Row

print("\n" + "=" * 100)
print("📊 SILVER LAYER TRANSFORMATION SUMMARY")
print("=" * 100)

# Calculate statistics
success_count = builtins.sum(1 for r in transformation_results if r["status"] == "success")
failed_count = builtins.sum(1 for r in transformation_results if r["status"] == "failed")
total_rows = builtins.sum(r["row_count"] for r in transformation_results)
total_duration = builtins.sum(r["duration_seconds"] for r in transformation_results)

dimensions_success = builtins.sum(1 for r in transformation_results if r.get("type") == "dimension" and r["status"] == "success")
facts_success = builtins.sum(1 for r in transformation_results if r.get("type") == "fact" and r["status"] == "success")

print(f"\nTransformation Results:")
print(f"  Total Tables Processed: {len(transformation_results)}")
print(f"  ✅ Successful: {success_count} ({dimensions_success} dimensions, {facts_success} facts)")
print(f"  ❌ Failed: {failed_count}")
print(f"  📊 Total Rows Created: {total_rows:,}")
print(f"  ⏱️  Total Duration: {total_duration:.2f} seconds")

# Display detailed results
if transformation_results:
    summary_df = spark.createDataFrame([Row(**r) for r in transformation_results])
    print("\nDetailed Transformation Results:")
    display(summary_df.orderBy("type", "status", "table"))

# Validate Silver schema
print("\n" + "=" * 100)
print("🔍 VALIDATING SILVER SCHEMA")
print("=" * 100)

silver_tables = spark.sql(f"SHOW TABLES IN {SILVER_SCHEMA}").collect()
print(f"\nSilver Layer Tables Created: {len(silver_tables)}")
for table in silver_tables:
    table_name = table.tableName
    try:
        count = spark.table(f"{SILVER_SCHEMA}.{table_name}").count()
        print(f"  ✅ {table_name:30} {count:>15,} rows")
    except Exception as e:
        print(f"  ❌ {table_name:30} Error: {str(e)[:50]}")

# Final status
print("\n" + "=" * 100)
if failed_count == 0:
    print("✅ SILVER LAYER TRANSFORMATION COMPLETED SUCCESSFULLY")
    print("=" * 100)
    print(f"Completion Time: {datetime.now()}")
    print(f"\nReady for Gold Layer transformation:")
    print(f"  - All {dimensions_success} dimensions available")
    print(f"  - All {facts_success} fact tables available")
    print(f"  - Total {total_rows:,} rows in Silver layer")
    dbutils.notebook.exit("SUCCESS")
else:
    print("⚠️ SILVER LAYER TRANSFORMATION COMPLETED WITH ERRORS")
    print("=" * 100)
    print(f"Completion Time: {datetime.now()}")
    print(f"\n{failed_count} table(s) failed to transform")
    print("Please review errors above and retry failed transformations")
    dbutils.notebook.exit(f"PARTIAL_SUCCESS: {failed_count} tables failed")